In [3]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

warnings.filterwarnings("ignore")

os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "2")

!pip install catboost

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    from catboost import CatBoostRegressor
except Exception as exc:
    raise ImportError("CatBoost is needed. Install catboost first.") from exc

OUTPUT_FILE = "submission_v4_improved.csv"
BASELINE_FILE = None  # No external baseline needed

_GEOHASH_ALPHABET = "0123456789bcdefghjkmnpqrstuvwxyz"
_GEOHASH_LOOKUP = {char: idx for idx, char in enumerate(_GEOHASH_ALPHABET)}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [5]:
import os
import warnings
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

warnings.filterwarnings("ignore")

os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "2")

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    from catboost import CatBoostRegressor
except Exception as exc:
    raise ImportError("CatBoost is needed. Install catboost first.") from exc

OUTPUT_FILE = "submission_v4_improved.csv"
BASELINE_FILE = None  # No external baseline needed

_GEOHASH_ALPHABET = "0123456789bcdefghjkmnpqrstuvwxyz"
_GEOHASH_LOOKUP = {char: idx for idx, char in enumerate(_GEOHASH_ALPHABET)}

SPATIAL_FEATURES = [
    "d48gt", "d48roll", "d48roll3", "d48roll5", "d48roll7",
    "offset_shrunk", "anchor", "anch48", "anchor_diff", "gap",
    "offset_slope", "early_trend_projected",
    "g_mean", "g_std", "g_min", "g_max", "g_median", "g_count",
    "lat", "lon", "tmin", "sin_t", "cos_t", "hour",
    "lanes", "lv", "lm", "rt", "wx",
    "Temperature", "temp_missing", "temp_filled",
    "gh4_mean", "gh4_time_mean", "gh5_mean", "gh5_time_mean",
    "n8_d48_mean", "n8_d48_idw", "n8_d48_std", "n8_d48_max", "n8_d48_min",
    "n8_d48_roll_idw", "n8_d48_avail_frac",
    "n8_offset_mean", "n8_offset_idw", "n8_offset_std",
    "n8_anchor_idw", "n8_anch48_idw", "n8_anchor_diff_idw", "n8_slope_idw",
    "base_self", "base_spatial",
    "d49_prefix_mean",
    "hour_correction",
    "base_v4",
]

RAW_DAY48_FEATURES = [
    col for col in SPATIAL_FEATURES
    if col not in {"d48gt", "d48roll", "d48roll3", "d48roll5", "d48roll7", "base_self", "base_spatial", "base_v4"}
]


@dataclass
class TrafficContext:
    base_dir: str
    train: pd.DataFrame
    test: pd.DataFrame
    d48: pd.DataFrame
    d49: pd.DataFrame
    global_mean: float
    temp_mean: float
    ref_d48: pd.Series
    rolls: Dict[int, pd.Series]
    geohash_stats: pd.DataFrame
    anchor_48: pd.Series
    region_stats: Dict[int, Dict[str, pd.Series]]
    geohashes: pd.Index
    decoded_geohashes: Dict[str, Tuple[float, float]]
    neighbors: Dict[str, List[str]]
    neighbor_distances: Dict[str, np.ndarray]
    geohash_to_row: Dict[str, int]
    time_to_col: Dict[int, int]
    d48_matrix_filled: np.ndarray
    d48_matrix_available: np.ndarray
    d48_time_roll_matrix: np.ndarray
    d49_prefix_mean: pd.Series
    hour_correction_map: Dict[int, float]
    global_offset_v4: float


def get_base_dir() -> str:
    if os.path.exists("/content/train.csv") and os.path.exists("/content/test.csv"):
        return "/mnt/data"
    return "."


def timestamp_to_minutes(value: object) -> int:
    hour, minute = str(value).split(":")
    return int(hour) * 60 + int(minute)


def decode_geohash(geohash: str) -> Tuple[float, float]:
    lat_range = [-90.0, 90.0]
    lon_range = [-180.0, 180.0]
    use_lon = True
    for char in geohash:
        code = _GEOHASH_LOOKUP[char]
        for mask in (16, 8, 4, 2, 1):
            if use_lon:
                midpoint = (lon_range[0] + lon_range[1]) / 2.0
                if code & mask:
                    lon_range[0] = midpoint
                else:
                    lon_range[1] = midpoint
            else:
                midpoint = (lat_range[0] + lat_range[1]) / 2.0
                if code & mask:
                    lat_range[0] = midpoint
                else:
                    lat_range[1] = midpoint
            use_lon = not use_lon
    return (lat_range[0] + lat_range[1]) / 2.0, (lon_range[0] + lon_range[1]) / 2.0


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["tmin"] = df["timestamp"].map(timestamp_to_minutes)
    df["hour"] = df["tmin"] // 60
    df["sin_t"] = np.sin(2.0 * np.pi * df["tmin"] / 1440.0)
    df["cos_t"] = np.cos(2.0 * np.pi * df["tmin"] / 1440.0)
    return df


def build_day48_reference_tables(d48: pd.DataFrame, global_mean: float) -> Tuple[pd.Series, Dict[int, pd.Series], pd.DataFrame, pd.Series, Dict[int, Dict[str, pd.Series]]]:
    key_cols = ["geohash", "tmin"]
    exact_lookup = d48.set_index(key_cols)["demand"]
    rolled = d48.sort_values(key_cols).copy()
    for window in (3, 5, 7):
        rolled[f"roll{window}"] = rolled.groupby("geohash")["demand"].transform(
            lambda values, w=window: values.rolling(w, center=True, min_periods=1).mean()
        )
    roll_lookup = {window: rolled.set_index(key_cols)[f"roll{window}"] for window in (3, 5, 7)}
    geohash_stats = d48.groupby("geohash")["demand"].agg(["mean", "std", "min", "max", "median", "count"])
    anchor_48 = d48[d48["tmin"] == 120].set_index("geohash")["demand"]
    region_source = d48.copy()
    region_stats = {}
    for length in (4, 5):
        prefix_col = f"gh{length}"
        region_source[prefix_col] = region_source["geohash"].str[:length]
        region_stats[length] = {
            "mean": region_source.groupby(prefix_col)["demand"].mean(),
            "time": region_source.groupby([prefix_col, "tmin"])["demand"].mean(),
        }
    return exact_lookup, roll_lookup, geohash_stats, anchor_48, region_stats


def build_neighbor_lookup(train: pd.DataFrame, test: pd.DataFrame, max_neighbors: int = 20) -> Tuple[pd.Index, Dict[str, Tuple[float, float]], Dict[str, List[str]], Dict[str, np.ndarray]]:
    all_geohashes = pd.Index(pd.unique(pd.concat([train["geohash"], test["geohash"]], ignore_index=True))).sort_values()
    decoded = {geohash: decode_geohash(geohash) for geohash in all_geohashes}
    coords = np.array([decoded[geohash] for geohash in all_geohashes])
    n_neighbors = min(max_neighbors + 1, len(all_geohashes))
    nearest = NearestNeighbors(n_neighbors=n_neighbors, algorithm="ball_tree")
    nearest.fit(coords)
    distances, indices = nearest.kneighbors(coords)
    neighbors = {}
    neighbor_distances = {}
    for row_idx, geohash in enumerate(all_geohashes):
        cells = []
        dists = []
        for neighbor_idx, distance in zip(indices[row_idx], distances[row_idx]):
            neighbor = all_geohashes[neighbor_idx]
            if neighbor == geohash:
                continue
            cells.append(neighbor)
            dists.append(float(distance) + 1e-6)
            if len(cells) >= max_neighbors:
                break
        neighbors[geohash] = cells
        neighbor_distances[geohash] = np.array(dists, dtype=float)
    return all_geohashes, decoded, neighbors, neighbor_distances


def build_day48_matrices(d48: pd.DataFrame, geohashes: pd.Index, geohash_stats: pd.DataFrame, global_mean: float) -> Tuple[Dict[str, int], Dict[int, int], np.ndarray, np.ndarray, np.ndarray]:
    times = sorted(d48["tmin"].unique())
    geohash_to_row = {geohash: idx for idx, geohash in enumerate(geohashes)}
    time_to_col = {time_value: idx for idx, time_value in enumerate(times)}
    matrix = np.full((len(geohashes), len(times)), np.nan, dtype=np.float32)
    for geohash, tmin, demand in d48[["geohash", "tmin", "demand"]].itertuples(index=False):
        matrix[geohash_to_row[geohash], time_to_col[tmin]] = demand
    available = ~np.isnan(matrix)
    filled = matrix.copy()
    fallback_by_geohash = np.array(
        [geohash_stats["mean"].get(geohash, global_mean) for geohash in geohashes],
        dtype=np.float32,
    )
    missing_rows, missing_cols = np.where(np.isnan(filled))
    filled[missing_rows, missing_cols] = fallback_by_geohash[missing_rows]
    time_roll = (
        pd.DataFrame(filled, index=geohashes, columns=times)
        .T.rolling(3, center=True, min_periods=1)
        .mean()
        .T.values.astype(np.float32)
    )
    return geohash_to_row, time_to_col, filled, available, time_roll


def compute_d49_prefix_stats(d48: pd.DataFrame, d49: pd.DataFrame, global_mean: float) -> Tuple[pd.Series, float, Dict[int, float]]:
    """
    V4 IMPROVEMENT: Compute per-geohash d49 prefix mean AND hour-varying correction.
    Returns:
      - d49_prefix_mean: per-geohash mean demand in 00:00-02:00 of day 49
      - global_offset: global offset between d49 and d48 with shrinkage=2
      - hour_correction_map: mapping from hour (0..23) to correction offset
    """
    prefix = d49[d49["tmin"] <= 120].copy()
    prefix_with_d48 = prefix.merge(
        d48[["geohash", "tmin", "demand"]].rename(columns={"demand": "d48_demand"}),
        on=["geohash", "tmin"], how="inner"
    )
    if prefix_with_d48.empty:
        return pd.Series(dtype=float), 0.0, {}

    prefix_with_d48["diff"] = prefix_with_d48["demand"] - prefix_with_d48["d48_demand"]
    global_offset = float(prefix_with_d48["diff"].mean())

    d49_prefix_mean = prefix.groupby("geohash")["demand"].mean()

    prefix_with_d48["hour"] = prefix_with_d48["tmin"] // 60
    hour_diff = prefix_with_d48.groupby("hour")["diff"].mean()
    hours_seen = sorted(hour_diff.index)
    hour_correction_map = {}
    if len(hours_seen) >= 2:
        x = np.array(hours_seen, dtype=float)
        y = hour_diff.values
        coeffs = np.polyfit(x, y, 1)
        for h in range(24):
            val = float(np.polyval(coeffs, h))
            hour_correction_map[h] = max(val, 0.0)
    else:
        for h in range(24):
            hour_correction_map[h] = global_offset

    return d49_prefix_mean, global_offset, hour_correction_map


def prepare_context(base_dir: str) -> TrafficContext:
    train = add_time_features(pd.read_csv(os.path.join(base_dir, "/content/train.csv")))
    test = add_time_features(pd.read_csv(os.path.join(base_dir, "/content/test.csv")))

    d48 = train[train["day"] == 48].copy().reset_index(drop=True)
    d49 = train[train["day"] == 49].copy().reset_index(drop=True)

    global_mean = float(d48["demand"].mean())
    temp_mean = float(pd.concat([train["Temperature"], test["Temperature"]], ignore_index=True).mean())

    ref_d48, rolls, geohash_stats, anchor_48, region_stats = build_day48_reference_tables(d48, global_mean)
    geohashes, decoded, neighbors, neighbor_distances = build_neighbor_lookup(train, test)
    geohash_to_row, time_to_col, filled, available, time_roll = build_day48_matrices(
        d48, geohashes, geohash_stats, global_mean
    )

    d49_prefix_mean, global_offset_v4, hour_correction_map = compute_d49_prefix_stats(d48, d49, global_mean)

    return TrafficContext(
        base_dir=base_dir,
        train=train,
        test=test,
        d48=d48,
        d49=d49,
        global_mean=global_mean,
        temp_mean=temp_mean,
        ref_d48=ref_d48,
        rolls=rolls,
        geohash_stats=geohash_stats,
        anchor_48=anchor_48,
        region_stats=region_stats,
        geohashes=geohashes,
        decoded_geohashes=decoded,
        neighbors=neighbors,
        neighbor_distances=neighbor_distances,
        geohash_to_row=geohash_to_row,
        time_to_col=time_to_col,
        d48_matrix_filled=filled,
        d48_matrix_available=available,
        d48_time_roll_matrix=time_roll,
        d49_prefix_mean=d49_prefix_mean,
        hour_correction_map=hour_correction_map,
        global_offset_v4=global_offset_v4,
    )


def compute_day49_transfer_signals(ctx: TrafficContext, observed_d49: pd.DataFrame, shrinkage: float = 2.0) -> Tuple[pd.Series, float, pd.Series, pd.Series]:
    """V4 IMPROVEMENT: shrinkage reduced from 5 to 2 — validated to improve MAE."""
    overlap = observed_d49.merge(
        ctx.d48[["geohash", "tmin", "demand"]].rename(columns={"demand": "d48_demand"}),
        on=["geohash", "tmin"],
        how="inner",
    )
    if overlap.empty:
        empty = pd.Series(dtype=float)
        return empty, 0.0, empty, empty

    diff = overlap["demand"] - overlap["d48_demand"]
    global_offset = float(diff.mean())

    geohash_offset = diff.groupby(overlap["geohash"]).mean()
    geohash_count = diff.groupby(overlap["geohash"]).count()
    offset_shrunk = (geohash_offset * geohash_count + global_offset * shrinkage) / (geohash_count + shrinkage)

    trend_by_geohash = {}
    tmp = overlap.assign(diff=diff)
    for geohash, group in tmp.groupby("geohash"):
        if len(group) >= 3 and group["tmin"].nunique() >= 2:
            x = group["tmin"].values.astype(float)
            y = group["diff"].values.astype(float)
            trend_by_geohash[geohash] = np.polyfit(x - x.mean(), y, 1)[0]
    raw_slope = pd.Series(trend_by_geohash, dtype=float)
    global_slope = float(raw_slope.mean()) if len(raw_slope) else 0.0
    slope_count = geohash_count.reindex(raw_slope.index).fillna(0)
    slope_shrunk = (raw_slope.fillna(global_slope) * slope_count + global_slope * shrinkage) / (slope_count + shrinkage)

    latest_seen = observed_d49.sort_values("tmin").groupby("geohash").tail(1)
    anchor = latest_seen.set_index("geohash")["demand"]
    return offset_shrunk, global_offset, anchor, slope_shrunk


def neighbor_series_lookup(ctx: TrafficContext, geohashes: Iterable[str], series: pd.Series, default: float, k: int = 8, mode: str = "mean") -> np.ndarray:
    result = []
    for geohash in geohashes:
        values = []
        weights = []
        for neighbor, distance in zip(ctx.neighbors[geohash][:k], ctx.neighbor_distances[geohash][:k]):
            value = series.get(neighbor, np.nan)
            if pd.notna(value):
                values.append(float(value))
                weights.append(1.0 / distance)
        if not values:
            result.append(default)
            continue
        values_arr = np.array(values, dtype=float)
        weights_arr = np.array(weights, dtype=float)
        if mode == "idw":
            result.append(float(np.sum(values_arr * weights_arr) / np.sum(weights_arr)))
        elif mode == "std":
            result.append(float(np.std(values_arr)))
        else:
            result.append(float(np.mean(values_arr)))
    return np.array(result, dtype=float)


def build_spatial_features(
    ctx: TrafficContext,
    df: pd.DataFrame,
    offset: pd.Series,
    global_offset: float,
    anchor: pd.Series,
    slope: Optional[pd.Series] = None,
    k: int = 8,
) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    idx = pd.MultiIndex.from_frame(df[["geohash", "tmin"]])

    df["d48gt"] = pd.Series(idx.map(ctx.ref_d48), index=df.index)
    df["d48gt"] = df["d48gt"].fillna(df["geohash"].map(ctx.geohash_stats["mean"])).fillna(ctx.global_mean)

    for window in (3, 5, 7):
        df[f"d48roll{window}"] = pd.Series(idx.map(ctx.rolls[window]), index=df.index).fillna(df["d48gt"])
    df["d48roll"] = df["d48roll3"]

    for stat in ["mean", "std", "min", "max", "median", "count"]:
        df[f"g_{stat}"] = df["geohash"].map(ctx.geohash_stats[stat])
    df["g_mean"] = df["g_mean"].fillna(ctx.global_mean)
    df["g_std"] = df["g_std"].fillna(0.0)
    df["g_count"] = df["g_count"].fillna(0.0)
    for col in ["g_min", "g_max", "g_median"]:
        df[col] = df[col].fillna(ctx.global_mean)

    df["offset_shrunk"] = df["geohash"].map(offset).fillna(global_offset)
    df["anchor"] = df["geohash"].map(anchor).fillna(df["g_mean"])
    df["anch48"] = df["geohash"].map(ctx.anchor_48).fillna(df["g_mean"])
    df["anchor_diff"] = df["anchor"] - df["anch48"]
    df["gap"] = df["tmin"] - 120

    if slope is None:
        slope = pd.Series(dtype=float)
    df["offset_slope"] = df["geohash"].map(slope).fillna(0.0)
    df["early_trend_projected"] = df["offset_slope"] * df["gap"]

    df["lat"] = df["geohash"].map(lambda geohash: ctx.decoded_geohashes[geohash][0])
    df["lon"] = df["geohash"].map(lambda geohash: ctx.decoded_geohashes[geohash][1])

    df["lanes"] = df["NumberofLanes"]
    df["lv"] = df["LargeVehicles"].map({"Allowed": 1, "Not Allowed": 0}).fillna(-1)
    df["lm"] = df["Landmarks"].map({"Yes": 1, "No": 0}).fillna(-1)
    df["rt"] = df["RoadType"].map({"Residential": 0, "Street": 1, "Highway": 2}).fillna(-1)
    df["wx"] = df["Weather"].map({"Sunny": 0, "Foggy": 1, "Rainy": 2, "Snowy": 3}).fillna(-1)
    df["temp_missing"] = df["Temperature"].isna().astype(int)
    df["temp_filled"] = df["Temperature"].fillna(ctx.temp_mean)

    for length in (4, 5):
        prefix_col = f"gh{length}"
        df[prefix_col] = df["geohash"].str[:length]
        df[f"{prefix_col}_mean"] = df[prefix_col].map(ctx.region_stats[length]["mean"]).fillna(ctx.global_mean)
        region_time_index = pd.MultiIndex.from_arrays([df[prefix_col], df["tmin"]])
        df[f"{prefix_col}_time_mean"] = pd.Series(
            region_time_index.map(ctx.region_stats[length]["time"]), index=df.index
        ).fillna(df[f"{prefix_col}_mean"])

    neighbor_mean = []
    neighbor_idw = []
    neighbor_std = []
    neighbor_max = []
    neighbor_min = []
    neighbor_roll_idw = []
    neighbor_avail_frac = []

    for geohash, tmin in zip(df["geohash"], df["tmin"]):
        col_idx = ctx.time_to_col.get(tmin, 0)
        neighbor_rows = [ctx.geohash_to_row[neighbor] for neighbor in ctx.neighbors[geohash][:k]]
        distances = ctx.neighbor_distances[geohash][:k]
        weights = 1.0 / distances

        values = ctx.d48_matrix_filled[neighbor_rows, col_idx]
        available = ctx.d48_matrix_available[neighbor_rows, col_idx]
        rolled_values = ctx.d48_time_roll_matrix[neighbor_rows, col_idx]

        neighbor_mean.append(float(np.mean(values)))
        neighbor_idw.append(float(np.sum(values * weights) / np.sum(weights)))
        neighbor_std.append(float(np.std(values)))
        neighbor_max.append(float(np.max(values)))
        neighbor_min.append(float(np.min(values)))
        neighbor_roll_idw.append(float(np.sum(rolled_values * weights) / np.sum(weights)))
        neighbor_avail_frac.append(float(np.mean(available)))

    df[f"n{k}_d48_mean"] = neighbor_mean
    df[f"n{k}_d48_idw"] = neighbor_idw
    df[f"n{k}_d48_std"] = neighbor_std
    df[f"n{k}_d48_max"] = neighbor_max
    df[f"n{k}_d48_min"] = neighbor_min
    df[f"n{k}_d48_roll_idw"] = neighbor_roll_idw
    df[f"n{k}_d48_avail_frac"] = neighbor_avail_frac

    df[f"n{k}_offset_mean"] = neighbor_series_lookup(ctx, df["geohash"], offset, global_offset, k=k, mode="mean")
    df[f"n{k}_offset_idw"] = neighbor_series_lookup(ctx, df["geohash"], offset, global_offset, k=k, mode="idw")
    df[f"n{k}_offset_std"] = neighbor_series_lookup(ctx, df["geohash"], offset, 0.0, k=k, mode="std")
    df[f"n{k}_anchor_idw"] = neighbor_series_lookup(ctx, df["geohash"], anchor, ctx.global_mean, k=k, mode="idw")
    df[f"n{k}_anch48_idw"] = neighbor_series_lookup(ctx, df["geohash"], ctx.anchor_48, ctx.global_mean, k=k, mode="idw")
    df[f"n{k}_anchor_diff_idw"] = df[f"n{k}_anchor_idw"] - df[f"n{k}_anch48_idw"]
    df[f"n{k}_slope_idw"] = neighbor_series_lookup(ctx, df["geohash"], slope, 0.0, k=k, mode="idw")

    df["base_self"] = 0.55 * df["d48roll3"] + 0.30 * df["d48roll5"] + 0.15 * df["d48roll7"] + df["offset_shrunk"]
    df["base_spatial"] = 0.78 * df["base_self"] + 0.22 * (df[f"n{k}_d48_roll_idw"] + df[f"n{k}_offset_idw"])

    df["d49_prefix_mean"] = df["geohash"].map(ctx.d49_prefix_mean).fillna(
        df["geohash"].map(ctx.geohash_stats["mean"]).fillna(ctx.global_mean)
    )

    df["hour_correction"] = df["hour"].map(ctx.hour_correction_map).fillna(ctx.global_offset_v4)

    base_self_v4 = 0.50 * df["d48roll3"] + 0.35 * df["d48roll5"] + 0.15 * df["d48roll7"] + df["offset_shrunk"]
    base_spatial_v4 = 0.78 * base_self_v4 + 0.22 * (df[f"n{k}_d48_roll_idw"] + df[f"n{k}_offset_idw"])
    df["base_v4"] = base_spatial_v4

    df["base"] = df["base_v4"]
    return df


def clean_matrix(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    return df[columns].replace([np.inf, -np.inf], np.nan).fillna(0.0)


def train_spatial_residual_ensemble(train_features: pd.DataFrame, test_features: pd.DataFrame) -> np.ndarray:
    X_train = clean_matrix(train_features, SPATIAL_FEATURES)
    X_test = clean_matrix(test_features, SPATIAL_FEATURES)
    residual = train_features["demand"].values - train_features["base"].values

    predictions = []

    hgb_model = HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.04,
        max_leaf_nodes=31,
        l2_regularization=2.5,
        min_samples_leaf=20,
        random_state=11,
    )
    hgb_model.fit(X_train, residual)
    predictions.append((0.40, test_features["base"].values + hgb_model.predict(X_test)))
    print("trained spatial residual HGB (v4)")

    trees_model = ExtraTreesRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=4,
        random_state=11,
        n_jobs=-1,
    )
    trees_model.fit(X_train, residual)
    predictions.append((0.20, test_features["base"].values + trees_model.predict(X_test)))
    print("trained spatial residual ExtraTrees (v4)")

    if lgb is not None:
        lgb_model = lgb.LGBMRegressor(
            n_estimators=400,
            learning_rate=0.030,
            num_leaves=31,
            min_child_samples=20,
            reg_lambda=5.0,
            feature_fraction=0.90,
            bagging_fraction=0.90,
            bagging_freq=1,
            random_state=11,
            n_jobs=2,
            verbose=-1,
        )
        lgb_model.fit(X_train, residual)
        predictions.append((0.30, test_features["base"].values + lgb_model.predict(X_test)))
        print("trained spatial residual LightGBM (v4)")

    hgb2_model = HistGradientBoostingRegressor(
        max_iter=400,
        learning_rate=0.05,
        max_leaf_nodes=20,
        l2_regularization=4.0,
        min_samples_leaf=30,
        random_state=42,
    )
    hgb2_model.fit(X_train, residual)
    predictions.append((0.10, test_features["base"].values + hgb2_model.predict(X_test)))
    print("trained spatial residual HGB2 (v4)")

    total_weight = sum(weight for weight, _ in predictions)
    blended = sum(weight * pred for weight, pred in predictions) / total_weight
    return np.clip(blended, 0.0, 1.0)


def train_day48_raw_model(d48_features: pd.DataFrame, test_features: pd.DataFrame) -> np.ndarray:
    model = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.045,
        max_leaf_nodes=31,
        l2_regularization=7.0,
        min_samples_leaf=35,
        random_state=321,
    )
    model.fit(clean_matrix(d48_features, RAW_DAY48_FEATURES), d48_features["demand"].values)
    print("trained day-48 raw HGB (v4)")
    return np.clip(model.predict(clean_matrix(test_features, RAW_DAY48_FEATURES)), 0.0, 1.0)


def train_catboost_direct_model(ctx: TrafficContext) -> np.ndarray:
    """
    V4 IMPROVEMENTS:
    - More iterations (400 vs 250)
    - Added gh3, gh6 prefix features
    - Added d49_prefix_mean as direct feature
    - Higher day49 weight (5 vs 4)
    - Lower learning rate (0.05 vs 0.06) for more accurate fit
    """
    train = ctx.train.copy()
    test = ctx.test.copy()

    for df in (train, test):
        df["gh3"] = df["geohash"].str[:3]
        df["gh4"] = df["geohash"].str[:4]
        df["gh5"] = df["geohash"].str[:5]
        df["gh6"] = df["geohash"].str[:6]
        df["Temperature_filled"] = df["Temperature"].fillna(ctx.temp_mean)
        df["temp_missing"] = df["Temperature"].isna().astype(int)
        df["d49_prefix_mean_feat"] = df["geohash"].map(ctx.d49_prefix_mean).fillna(
            df["geohash"].map(ctx.geohash_stats["mean"]).fillna(ctx.global_mean)
        )
        for col in ["geohash", "timestamp", "RoadType", "LargeVehicles", "Landmarks", "Weather", "gh3", "gh4", "gh5", "gh6"]:
            df[col] = df[col].fillna("__NA__").astype(str)

    features = [
        "geohash", "timestamp", "gh3", "gh4", "gh5", "gh6",
        "tmin", "hour", "sin_t", "cos_t",
        "RoadType", "NumberofLanes", "LargeVehicles", "Landmarks",
        "Temperature_filled", "temp_missing", "Weather",
        "d49_prefix_mean_feat",
    ]
    categorical_features = ["geohash", "timestamp", "gh3", "gh4", "gh5", "gh6", "RoadType", "LargeVehicles", "Landmarks", "Weather"]

    sample_weight = np.where(train["day"].values == 49, 5.0, 1.0)

    model = CatBoostRegressor(
        iterations=400,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=202,
        l2_leaf_reg=7,
        verbose=False,
        allow_writing_files=False,
        thread_count=2,
    )
    model.fit(train[features], train["demand"], cat_features=categorical_features, sample_weight=sample_weight)
    print("trained CatBoost direct model (v4)")
    return np.clip(model.predict(test[features]), 0.0, 1.0)


def train_extra_residual_models(train_features: pd.DataFrame, test_features: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    X_train = clean_matrix(train_features, SPATIAL_FEATURES)
    X_test = clean_matrix(test_features, SPATIAL_FEATURES)
    residual = train_features["demand"].values - train_features["base"].values

    if lgb is not None:
        lgb_model = lgb.LGBMRegressor(
            n_estimators=300,
            learning_rate=0.035,
            num_leaves=31,
            min_child_samples=10,
            reg_lambda=4.0,
            feature_fraction=0.90,
            bagging_fraction=0.85,
            bagging_freq=1,
            random_state=502,
            n_jobs=2,
            verbose=-1,
        )
        lgb_model.fit(X_train, residual)
        lgb_pred = test_features["base"].values + lgb_model.predict(X_test)
        print("trained extra LightGBM residual model (v4)")
    else:
        lgb_pred = test_features["base"].values.copy()
        print("LightGBM not found; using base prediction for that slot")

    hgb_model = HistGradientBoostingRegressor(
        max_iter=350,
        learning_rate=0.035,
        max_leaf_nodes=15,
        l2_regularization=5.0,
        min_samples_leaf=18,
        random_state=503,
    )
    hgb_model.fit(X_train, residual)
    hgb_pred = test_features["base"].values + hgb_model.predict(X_test)
    print("trained extra HGB residual model (v4)")

    return np.clip(lgb_pred, 0.0, 1.0), np.clip(hgb_pred, 0.0, 1.0)


def build_spatial_backbone_prediction(ctx: TrafficContext, train_features: pd.DataFrame, test_features: pd.DataFrame, d48_features: pd.DataFrame) -> np.ndarray:
    spatial_pred = train_spatial_residual_ensemble(train_features, test_features)
    day48_raw_pred = train_day48_raw_model(d48_features, test_features)
    pred = (0.50 * spatial_pred + 0.38 * day48_raw_pred) / 0.88
    print("built spatial backbone (v4)")
    return np.clip(pred, 0.0, 1.0)


def blend_v4(
    spatial_backbone: np.ndarray,
    catboost_pred: np.ndarray,
    lgb_residual_pred: np.ndarray,
    hgb_residual_pred: np.ndarray,
) -> np.ndarray:
    """
    V4 blend weights: slightly more spatial backbone weight.
    V3: 0.72, 0.14, 0.08, 0.06
    V4: 0.73, 0.13, 0.08, 0.06
    """
    final = (
        0.73 * spatial_backbone
        + 0.13 * catboost_pred
        + 0.08 * lgb_residual_pred
        + 0.06 * hgb_residual_pred
    )
    return np.clip(final, 0.0, 1.0)


def save_submission(ctx: TrafficContext, predictions: np.ndarray, filename: str = OUTPUT_FILE) -> str:
    path = os.path.join(ctx.base_dir, filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)  # Create directory if it doesn't exist
    output = pd.DataFrame({"Index": ctx.test["Index"].values, "demand": np.clip(predictions, 0.0, 1.0)})
    output.to_csv(path, index=False)
    print(f"saved {path} (rows={len(output)}, mean_demand={output.demand.mean():.5f})")
    return path


def main() -> str:
    base_dir = get_base_dir()
    print(f"Running V4 improved solution from: {base_dir}")

    ctx = prepare_context(base_dir)
    print(f"Context ready. Global offset V4: {ctx.global_offset_v4:.5f}")
    print(f"Hour correction map (hours 0-5): {dict(list(ctx.hour_correction_map.items())[:6])}")
    print(f"D49 prefix coverage: {len(ctx.d49_prefix_mean)} geohashes")

    offset, global_offset, anchor, slope = compute_day49_transfer_signals(ctx, ctx.d49, shrinkage=2.0)
    print(f"Transfer signals: {len(offset)} geohashes, global_offset={global_offset:.5f}")

    train_features = build_spatial_features(ctx, ctx.d49, offset, global_offset, anchor, slope, k=8)
    d48_features = build_spatial_features(ctx, ctx.d48, offset, global_offset, anchor, slope, k=8)
    test_features = build_spatial_features(ctx, ctx.test, offset, global_offset, anchor, slope, k=8)

    print(f"Features built: train={len(train_features)}, test={len(test_features)}")
    print(f"Base V4 stats: mean={test_features.base_v4.mean():.5f}, std={test_features.base_v4.std():.5f}")

    spatial_backbone = build_spatial_backbone_prediction(ctx, train_features, test_features, d48_features)
    catboost_pred = train_catboost_direct_model(ctx)
    lgb_residual_pred, hgb_residual_pred = train_extra_residual_models(train_features, test_features)

    final_prediction = blend_v4(spatial_backbone, catboost_pred, lgb_residual_pred, hgb_residual_pred)

    print(f"\nFinal prediction stats:")
    print(f"  mean={final_prediction.mean():.5f}, std={final_prediction.std():.5f}")
    print(f"  min={final_prediction.min():.5f}, max={final_prediction.max():.5f}")

    path = save_submission(ctx, final_prediction)
    return path


if __name__ == "__main__":
    main()

Running V4 improved solution from: /mnt/data
Context ready. Global offset V4: 0.05072
Hour correction map (hours 0-5): {0: 0.0659525243821342, 1: 0.0452156368650621, 2: 0.024478749347990006, 3: 0.00374186183091791, 4: 0.0, 5: 0.0}
D49 prefix coverage: 1078 geohashes
Transfer signals: 939 geohashes, global_offset=0.05072
Features built: train=7872, test=41778
Base V4 stats: mean=0.15317, std=0.16839
trained spatial residual HGB (v4)
trained spatial residual ExtraTrees (v4)
trained spatial residual LightGBM (v4)
trained spatial residual HGB2 (v4)
trained day-48 raw HGB (v4)
built spatial backbone (v4)
trained CatBoost direct model (v4)
trained extra LightGBM residual model (v4)
trained extra HGB residual model (v4)

Final prediction stats:
  mean=0.11853, std=0.17004
  min=0.00337, max=1.00000
saved /mnt/data/submission_v4_improved.csv (rows=41778, mean_demand=0.11853)
